# ADNOC Approved Vendor List — 10.12.2025

Renders `data/bidders_details/ADNOC Approved Vendor List as of 10.12.2025.xlsx`.

One sheet, ~18k rows. The grain is **one row per (product group × vendor × manufacturer)** —
a vendor appears once for every product group it is approved in, and once per manufacturer
it represents in that group. So vendor names repeat by design.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import Markdown

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 200)

XLSX = (
    Path.home()
    / "git_workspace"
    / "procurement-and-cost-estimation"
    / "data"
    / "bidders_details"
    / "ADNOC Approved Vendor List as of 10.12.2025.xlsx"
)
assert XLSX.exists(), XLSX
print(XLSX.name, f"({XLSX.stat().st_size / 1024:,.0f} KB)")

ADNOC Approved Vendor List as of 10.12.2025.xlsx (855 KB)


## 1. Load

Header is on row 0, no title block. `Manufacture Number` is an identifier with blanks, so it
is read as nullable `Int64` rather than letting NaN force it to float.

In [2]:
avl = pd.read_excel(XLSX, sheet_name="Sheet1", engine="openpyxl")
avl.columns = [" ".join(str(c).split()) for c in avl.columns]

for c in ("Product Group Number", "Vendor Number", "Manufacture Number"):
    avl[c] = avl[c].astype("Int64")
for c in avl.select_dtypes("object"):
    avl[c] = avl[c].astype("string").str.strip()

print(f"{avl.shape[0]:,} rows x {avl.shape[1]} cols")
avl.head(20)

18,032 rows x 8 cols


,Product Group Number,Product Group Description,Vendor Number,Vendor Name,Manufacture Number,Manufacture name,Manufacture Country,Remarks
0,122010,A/C UNITS – SPLIT & WINDOW,10000238,SYSTEMS EQUIPMENT L.L.C-O.P.C,20002132,AXIS SOLUTIONS PVT LTD,India,<NA>
1,122010,A/C UNITS – SPLIT & WINDOW,10000101,U.T.S CARRIER LLC,20007034,CARRIER INTERNATIONAL CORPORATION,United States of America,Migrated
2,122010,A/C UNITS – SPLIT & WINDOW,10007311,S K M AIR CONDITIONING - L L C,20001799,SKM AIR CONDITIONING EQUIPMENT,United Arab Emirates,Migrated-SQM
3,122010,A/C UNITS – SPLIT & WINDOW,10000255,EMIRATES LINK TECHNOLOGY LLC,20009282,WAROM TECHNOLOGY INCORPORATED COMPA,China,<NA>
4,122010,A/C UNITS – SPLIT & WINDOW,10000149,AL YASEAH OIL & GAS INDUSTRY SUPPLI,20020456,Wozair Limited,United Kingdom,<NA>
5,122010,A/C UNITS – SPLIT & WINDOW,10068140,FLOW LINE CONTROLS OIL FIELD SUPPLI,20035015,TMS ENDUSTRIYEL SOGUTMA SANAYI TICA,Republic of Turkiye,<NA>
6,122010,A/C UNITS – SPLIT & WINDOW,10000020,United Technical Services Company,20010420,CARRIER INTERNATIONAL CORP.,United Arab Emirates,<NA>
7,122010,A/C UNITS – SPLIT & WINDOW,10000305,JOHNSON CONTROLS AIR CONDITIONING &,20002571,YORK INTERNATIONAL (JOHNSON CONTROL,United States of America,Migrated
8,122010,A/C UNITS – SPLIT & WINDOW,10070167,Stolway Manufacturing - Sole Propri,<NA>,( LOCAL MANUFACTURE ),<NA>,Ref SQM# WS191062592
9,122010,A/C UNITS – SPLIT & WINDOW,10008436,PETRA AIR CONDITIONING COMPANY - LL,20003882,PETRA ENGINEERING INDUSTRIES CO.,Jordan,<NA>


## 2. Schema and completeness

In [3]:
pd.DataFrame(
    {
        "dtype": avl.dtypes.astype(str),
        "non_null": avl.notna().sum(),
        "nulls": avl.isna().sum(),
        "null_pct": (avl.isna().mean() * 100).round(1),
        "unique": avl.nunique(dropna=True),
        "sample": [avl[c].dropna().iloc[0] if avl[c].notna().any() else None for c in avl.columns],
    }
)

,dtype,non_null,nulls,null_pct,unique,sample
Product Group Number,Int64,18032,0,0.0,1143,122010
Product Group Description,string,18032,0,0.0,1141,A/C UNITS – SPLIT & WINDOW
Vendor Number,Int64,18032,0,0.0,1346,10000238
Vendor Name,string,18032,0,0.0,1345,SYSTEMS EQUIPMENT L.L.C-O.P.C
Manufacture Number,Int64,16691,1341,7.4,4916,20002132
Manufacture name,string,18032,0,0.0,4850,AXIS SOLUTIONS PVT LTD
Manufacture Country,string,16681,1351,7.5,75,India
Remarks,string,5256,12776,70.9,1039,Migrated


## 3. Grain check

Confirm the composite key, and see how many rows are exact duplicates on it.

In [4]:
KEY = ["Product Group Number", "Vendor Number", "Manufacture Number"]

dup_key = avl.duplicated(KEY, keep=False) & avl["Manufacture Number"].notna()
print(f"rows                       {len(avl):,}")
print(f"distinct {'+'.join(k.split()[0] for k in KEY):<25} {avl[KEY].drop_duplicates().shape[0]:,}")
print(f"rows in dup key groups     {dup_key.sum():,}")
print(f"fully duplicated rows      {avl.duplicated(keep=False).sum():,}")

avl[dup_key].sort_values(KEY).head(20)

rows                       18,032
distinct Product+Vendor+Manufacture 18,030
rows in dup key groups     4
fully duplicated rows      2


,Product Group Number,Product Group Description,Vendor Number,Vendor Name,Manufacture Number,Manufacture name,Manufacture Country,Remarks
12346,324216,GENERAL ELECTRICAL ACCESSORIES & SPARES - SUPPLIERS,10075314,ERCOMA TRADING LLC,20033722,SICHUAN SUNLIGHT INTELLIGENT ELECTR,China,<NA>
12347,324216,GENERAL ELECTRICAL ACCESSORIES & SPARES - SUPPLIERS,10075314,ERCOMA TRADING LLC,20033722,SICHUAN SUNLIGHT INTELLIGENT ELECTR,China,<NA>
14397,344855,LEAK DETECTION SYSTEMS (FOR PIPES / PIPELINES),10000184,SUWAIDI ENGINEERING GROUP WLL,20033807,AP Sensing GmbH,Germany,<NA>
14398,344855,LEAK DETECTION SYSTEMS (FOR PIPES / PIPELINES),10000184,SUWAIDI ENGINEERING GROUP WLL,20033807,AP Sensing GmbH,Germany,WS224556252


In [5]:
# Cardinality of each entity behind the grain
pd.Series(
    {
        "product groups": avl["Product Group Number"].nunique(),
        "vendors": avl["Vendor Number"].nunique(),
        "manufacturers": avl["Manufacture Number"].nunique(),
        "manufacturer names": avl["Manufacture name"].nunique(),
        "countries": avl["Manufacture Country"].nunique(),
    },
    name="distinct",
).to_frame()

,distinct
product groups,1143
vendors,1346
manufacturers,4916
manufacturer names,4850
countries,75


In [6]:
# Numbers and names should be 1:1 — flag any code carrying more than one label
for num, name in [
    ("Product Group Number", "Product Group Description"),
    ("Vendor Number", "Vendor Name"),
    ("Manufacture Number", "Manufacture name"),
]:
    counts = avl.groupby(num, observed=True)[name].nunique()
    bad = counts[counts > 1]
    print(f"{num:<22} -> {len(bad)} code(s) with multiple {name!r} values")
    if len(bad):
        display(avl[avl[num].isin(bad.index)][[num, name]].drop_duplicates().sort_values(num).head(20))

Product Group Number   -> 0 code(s) with multiple 'Product Group Description' values
Vendor Number          -> 0 code(s) with multiple 'Vendor Name' values
Manufacture Number     -> 0 code(s) with multiple 'Manufacture name' values


## 4. Breakdowns

In [7]:
display(Markdown("**Manufacture Country** — top 25"))
country = avl["Manufacture Country"].value_counts(dropna=False).to_frame("rows")
country["pct"] = (country["rows"] / len(avl) * 100).round(1)
display(country.head(25))

display(Markdown("**Remarks**"))
display(avl["Remarks"].value_counts(dropna=False).to_frame("rows").head(20))

**Manufacture Country** — top 25

,rows,pct
Manufacture Country,,
Italy,2094,11.6
India,2036,11.3
China,1875,10.4
United States of America,1726,9.6
United Kingdom,1588,8.8
United Arab Emirates,1399,7.8
<NA>,1351,7.5
Germany,1171,6.5
South Korea,819,4.5


**Remarks**

,rows
Remarks,
<NA>,12776
Migrated,2333
APPROVED,141
PQ,118
Approved,92
Re-assessment regularization,59
Batch IV -BOROUGE,55
WS169490983,29
Additional PG located from SDM,29


In [8]:
display(Markdown("**Vendors with the widest approval footprint**"))
display(
    avl.groupby(["Vendor Number", "Vendor Name"], observed=True)
    .agg(
        product_groups=("Product Group Number", "nunique"),
        manufacturers=("Manufacture Number", "nunique"),
        rows=("Vendor Number", "size"),
    )
    .sort_values("product_groups", ascending=False)
    .head(25)
)

display(Markdown("**Product groups with the most approved vendors**"))
display(
    avl.groupby(["Product Group Number", "Product Group Description"], observed=True)
    .agg(vendors=("Vendor Number", "nunique"), manufacturers=("Manufacture Number", "nunique"))
    .sort_values("vendors", ascending=False)
    .head(25)
)

**Vendors with the widest approval footprint**

,,product_groups,manufacturers,rows
Vendor Number,Vendor Name,,,
10000275,PETRO MIDDLE EAST - L.L.C - S.P.C,220,84,378
10000152,MARJAN INDUSTRIAL DEVELOPMENT LLC,218,63,295
10000104,AL MOUSAWI TRADING CO. LLC,216,60,310
10000026,International Development Company L,209,79,334
10008003,EMDAD - L L C,203,94,300
10000261,LIWA PETROLEUM & INDUSTRIAL SUPPLIE,203,64,321
10000011,GULF AUTOMATION SERVICES & OIL FIEL,194,69,280
10000099,AL MASAOOD OIL INDUSTRY SUPPLIES &S,192,63,233
10000318,ABU DHABI OILFIELD SERVICES COMPANY,164,77,257


**Product groups with the most approved vendors**

,,vendors,manufacturers
Product Group Number,Product Group Description,,
141032,"PRESSURE VESSELS UPTO 35MM (CS/SS/AS/CLAD)""-U STAMP",110,106
182415,"VALVES - BALL - API 6D - UP TO 12""",107,145
182420,"VALVES - BALL - API 6D - ABOVE 12""",104,136
182405,VALVES - BALL - API 6A,89,112
182408,VALVES - BALL (PIPELINES) - API 6D,85,94
184620,VALVES - DOUBLE BLOCK & BLEED - API,83,100
182605,VALVES - BUTTERFLY-ALL TYPE - API 609 / BS EN 593,78,112
160744,FLANGES FOR PIPES - CS/AS/SS,78,110
148560,STEEL STRUCTURE FABRICATED,77,29


In [9]:
display(Markdown("**Single-source product groups** (exactly one approved vendor)"))
per_group = avl.groupby(["Product Group Number", "Product Group Description"], observed=True)[
    "Vendor Number"
].nunique()
single = per_group[per_group == 1]
print(f"{len(single):,} of {per_group.size:,} product groups have a single approved vendor")
display(single.to_frame("vendors").head(30))

**Single-source product groups** (exactly one approved vendor)

131 of 1,143 product groups have a single approved vendor


,,vendors
Product Group Number,Product Group Description,
141054,SPARE PARTS FOR DRYERS; EVAPORATORS; COLUMNS,1
141055,FUEL STATION,1
141056,SPARE PARTS FOR WEIGHTFEEDER & MECH.(Polymer specfic),1
141058,BLENDING; STORAGE; LOADING AND BUNKERING ( polymer specfic),1
141059,SPARE PARTS FOR VESSELS; HOPPERS; DRUMS,1
141069,ROTARY KILN PACKAGE,1
141070,TOOLS & REQUISITES FOR PRESSURE VESSELS CLEANING,1
142015,PUMPS - CENTRIFUGAL - JET,1
142093,VANE PUMP (PRODUCT & SPARE PARTS),1


## 5. Lookups

`search` matches any column; `vendor` and `group` pivot the list around one entity.

In [10]:
def search(term: str, df: pd.DataFrame | None = None) -> pd.DataFrame:
    """Case-insensitive substring match across all columns."""
    df = avl if df is None else df
    mask = (
        df.astype("string")
        .apply(lambda s: s.str.contains(term, case=False, na=False, regex=False))
        .any(axis=1)
    )
    return df[mask]


def vendor(term: str) -> pd.DataFrame:
    """Everything one vendor is approved for, by product group."""
    hit = avl[avl["Vendor Name"].str.contains(term, case=False, na=False, regex=False)]
    return hit.sort_values(["Vendor Name", "Product Group Description"])


def group(term: str) -> pd.DataFrame:
    """Every approved vendor for product groups matching `term`."""
    hit = avl[avl["Product Group Description"].str.contains(term, case=False, na=False, regex=False)]
    return hit.sort_values(["Product Group Description", "Vendor Name"])


group("VALVES - GATE").head(30)

,Product Group Number,Product Group Description,Vendor Number,Vendor Name,Manufacture Number,Manufacture name,Manufacture Country,Remarks
8731,184208,VALVES - GATE - API - THRU CONDUIT (FOR PIPELINES),10000042,ABU DHABI MARITIME & MERCANTILE INT,20006664,ADVANCED TECHNOLOGY VALVE S.P.A.,Italy,<NA>
8715,184208,VALVES - GATE - API - THRU CONDUIT (FOR PIPELINES),10000958,ABU DHABI VALVES & CONTROL ESTABLIS,20002422,HINDLE (PENTAIR FLOW CONTROLS MIDD,United Kingdom,<NA>
8720,184208,VALVES - GATE - API - THRU CONDUIT (FOR PIPELINES),10067491,ACEKROFT ENERGY SOLUTIONS- L.L.C,20008472,PANAM ENGINEERS LTD,India,<NA>
8730,184208,VALVES - GATE - API - THRU CONDUIT (FOR PIPELINES),10007923,AL MADAR OIL FIELD SUPPLIES & SERV,20000929,L&T VALVES LIMITED,India,<NA>
8734,184208,VALVES - GATE - API - THRU CONDUIT (FOR PIPELINES),10007923,AL MADAR OIL FIELD SUPPLIES & SERV,20034762,Ghatge Patil Industries Limited,India,<NA>
8705,184208,VALVES - GATE - API - THRU CONDUIT (FOR PIPELINES),10000027,AL MANSOORI INDUSTRIAL SUPPLY COMPA,20009237,VELAN INC.,Canada,<NA>
8693,184208,VALVES - GATE - API - THRU CONDUIT (FOR PIPELINES),10000588,AL THURAYYA TRADING - SOLE PROPRIET,20007558,FLUITEK ORSENIGO VALVES S.R.L,Italy,<NA>
8723,184208,VALVES - GATE - API - THRU CONDUIT (FOR PIPELINES),10000149,AL YASEAH OIL & GAS INDUSTRY SUPPLI,20009220,VALVOSIDER S.R.L.,United Arab Emirates,<NA>
8733,184208,VALVES - GATE - API - THRU CONDUIT (FOR PIPELINES),10000149,AL YASEAH OIL & GAS INDUSTRY SUPPLI,20001972,B.F.E.( BONNEY FORGE ) Srl,Italy,<NA>
8697,184208,VALVES - GATE - API - THRU CONDUIT (FOR PIPELINES),10000034,ALMANSOORI SPECIALIZED ENGINEERINGC,20001469,SEVERN GLOCON LTD.,United Kingdom,<NA>


In [11]:
vendor("AL MASAOOD").head(30)

,Product Group Number,Product Group Description,Vendor Number,Vendor Name,Manufacture Number,Manufacture name,Manufacture Country,Remarks
17901,846606,"VEHICLES - SALOON , STATION WAGONS & PICKUPS",10000284,AL MASAOOD AUTOMOBILES COMPANY L.L.,20008361,NISSAN,Japan,New Pre-Qualification
17525,841848,PORTACABIN (CARAVAN),10007915,AL MASAOOD BERGUM - L. L. C - O. P.,20010138,AL MASAOOD BERGUM COMPANY,United Arab Emirates,Migrated
1618,141532,FILTERS & AIR INTAKE SYSTEMS - FOR TURBINE,10000049,AL MASAOOD JOHN BROWN LIMITED - L L,20007983,JOHN BROWN ENGINEERING LTD (KVAERNE,United Kingdom,<NA>
17985,900001,MECHANICAL REPAIR REFURBISHMENT FACILITIES,10000049,AL MASAOOD JOHN BROWN LIMITED - L L,20007983,JOHN BROWN ENGINEERING LTD (KVAERNE,United Kingdom,<NA>
1596,141526,"PARTS FOR TURBINES -( BLADES, COMPONENTS ETC)",10000049,AL MASAOOD JOHN BROWN LIMITED - L L,20005776,G.E. (UK),United Kingdom,<NA>
1597,141526,"PARTS FOR TURBINES -( BLADES, COMPONENTS ETC)",10000049,AL MASAOOD JOHN BROWN LIMITED - L L,20007983,JOHN BROWN ENGINEERING LTD (KVAERNE,United Kingdom,<NA>
4404,147013,ACCESS FITTING (COUPONS /PROBES ETC) FOR CORROSION MONI...,10000035,AL MASAOOD L.L.C,20033556,TUBEFIT ENGINEERS,India,<NA>
1189,141064,AIR COOLERS-FINNED TUBED,10000035,AL MASAOOD L.L.C,20006910,BELLELI ENERGY S.R.L,Italy,<NA>
1231,141064,AIR COOLERS-FINNED TUBED,10000035,AL MASAOOD L.L.C,20006975,BRONSWERK HEAT TRANSFER B.V.,Netherlands,<NA>
3094,144054,AIR DRYERS,10000035,AL MASAOOD L.L.C,20007090,CIRMAC B.V.,Netherlands,<NA>


## 6. Export the cleaned frame (optional)

In [12]:
out = XLSX.parent / "adnoc_avl_clean.csv"
# avl.to_csv(out, index=False, encoding="utf-8-sig")
print(f"Uncomment above to write {out}")

Uncomment above to write C:\Users\irahu\git_workspace\procurement-and-cost-estimation\data\bidders_details\adnoc_avl_clean.csv
